In [5]:
import yfinance as yf

# Get and save the historical data for the following tickers
tickers = [
    "ADBE",
    "AMD",
    "AXP",
    "XOM",
    "FDX",
    "HSY",
    "JPM",
    "HPE",
    "MRNA",
    "NVDA"
]

for each in tickers:
    
    # get ticker object
    tickr = yf.Ticker(each)
    
    # get historical data from 2015 - 2025
    data = tickr.history(start='2015-01-01', end='2025-01-01')

    data.to_csv(f"../data/raw/{each}.csv")

    print(f"Saved {each} data to ../data/raw/{each}.csv")


Saved ADBE data to ../data/raw/ADBE.csv
Saved AMD data to ../data/raw/AMD.csv
Saved AXP data to ../data/raw/AXP.csv
Saved XOM data to ../data/raw/XOM.csv
Saved FDX data to ../data/raw/FDX.csv
Saved HSY data to ../data/raw/HSY.csv
Saved JPM data to ../data/raw/JPM.csv
Saved HPE data to ../data/raw/HPE.csv
Saved MRNA data to ../data/raw/MRNA.csv
Saved NVDA data to ../data/raw/NVDA.csv


In [25]:
import pandas as pd

# Load each ticker data as its own dataframe

data = {}

for each in tickers:

    data[each] = pd.read_csv(f"../data/raw/{each}.csv")

data["NVDA"].head()


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2015-01-02 00:00:00-05:00,0.482423,0.486018,0.474754,0.482423,113680000,0.0,0.0
1,2015-01-05 00:00:00-05:00,0.482423,0.483861,0.472118,0.474275,197952000,0.0,0.0
2,2015-01-06 00:00:00-05:00,0.474994,0.475473,0.459416,0.459896,197764000,0.0,0.0
3,2015-01-07 00:00:00-05:00,0.463251,0.467325,0.457259,0.458697,321808000,0.0,0.0
4,2015-01-08 00:00:00-05:00,0.463970,0.478828,0.463730,0.475952,283780000,0.0,0.0


In [23]:
import matplotlib.pyplot as plt

# Plot ADBE, MRNA and NVDA close for full series
plot_tickers = ["ADBE","MRNA","NVDA"]

fig, axs = plt.subplots(figsize=(12,8), nrows=3, ncols=1, sharex=True)

for idx, ax in enumerate(axs):
    df = data[plot_tickers[idx]]
    ax.plot(df.Date, df.Close, label=plot_tickers[idx])
    ax.set_title(plot_tickers[idx])

    ticks = df.index[[0, len(df)//2, -1]]
    ax.set_xticks(ticks)

fig.supxlabel('Date')
fig.supylabel('Close')

fig.tight_layout()
#plt.show()
plt.close()

In [61]:
# FEATURES
import numpy as np

for each in tickers:


    df = data[each]
    df["Ticker"] = each

    # momentum
    df["momentum_1m"] = df.Close / df.Close.shift(30) - 1
    df["momentum_3m"] = df.Close / df.Close.shift(90) - 1
    df["momentum_6m"] = df.Close / df.Close.shift(180) - 1
    df["momentum_12m"] = df.Close / df.Close.shift(360) - 1

    # compute the log returns and then volatility
    df["log_ret"] = np.log(df.Close / df.Close.shift(1))
    df["volatility"] = df["log_ret"].rolling(window=252).std() * np.sqrt(252)   # from formula sqrt(252) 

    # get rolling volume change
    df["volume_ratio"] = df.Volume / df.Volume.rolling(30).mean()

    # get fwd return 30 day
    df["target"] = df.Close.shift(-30) / df.Close - 1

In [ ]:
features = ["momentum_1m", "momentum_3m", "momentum_6m", "momentum_12m", "volatility", "volume_ratio"]

df_all = pd.concat(
    [df.assign(Ticker=ticker) for ticker, df in data.items()], ignore_index=True
)
df_all = df_all.dropna(subset=features + ["target"])

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,momentum_1m,momentum_3m,momentum_6m,momentum_12m,log_ret,volatility,Log_ret,Volatility,volume_ratio,target,Ticker
360,2016-06-08 00:00:00-04:00,98.449997,98.910004,97.989998,98.680000,2078700,0.0,0.0,0.026313,0.107147,0.184350,0.364114,0.001623,0.272227,0.001623,0.272227,1.003313,-0.023713,ADBE
361,2016-06-09 00:00:00-04:00,98.360001,98.699997,97.849998,98.070000,1609400,0.0,0.0,0.025408,0.094409,0.158398,0.362462,-0.006201,0.271566,-0.006201,0.271566,0.782761,-0.000102,ADBE
362,2016-06-10 00:00:00-04:00,96.889999,97.870003,96.750000,97.089996,1863800,0.0,0.0,0.036843,0.103923,0.135556,0.376577,-0.010043,0.271784,-0.010043,0.271784,0.931158,0.002369,ADBE
363,2016-06-13 00:00:00-04:00,96.870003,98.089996,96.699997,96.959999,2291500,0.0,0.0,0.029081,0.101943,0.155524,0.363521,-0.001340,0.271785,-0.001340,0.271785,1.161714,0.009798,ADBE
364,2016-06-14 00:00:00-04:00,96.529999,97.260002,96.320000,96.980003,1889800,0.0,0.0,0.021918,0.122974,0.166607,0.329951,0.000206,0.271479,0.000206,0.271479,0.965256,0.009796,ADBE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23930,2024-11-04 00:00:00-05:00,136.995932,138.743202,135.358491,135.837738,187528200,0.0,0.0,0.170222,0.097360,0.874034,2.597728,0.004789,0.511825,0.004789,0.511825,0.753894,-0.041536,NVDA
23931,2024-11-05 00:00:00-05:00,137.235558,140.151000,137.115750,139.691727,160537400,0.0,0.0,0.157525,0.132604,1.014917,2.519697,0.027977,0.511519,0.027977,0.511519,0.662652,-0.078558,NVDA
23932,2024-11-06 00:00:00-05:00,142.736953,146.261444,141.738514,145.382812,242043900,0.0,0.0,0.178933,0.171540,1.158543,2.704354,0.039932,0.512608,0.039932,0.512608,1.004985,-0.102472,NVDA
23933,2024-11-07 00:00:00-05:00,146.161630,148.697661,145.941972,148.647751,207323300,0.0,0.0,0.200258,0.213767,0.896050,2.802628,0.022209,0.512910,0.022209,0.512910,0.872323,-0.095182,NVDA


In [64]:
train = df_all[df_all.Date < "2022-01-01"]
valid = df_all[(df_all.Date >= "2022-01-01") & (df_all.Date < "2023-01-01")]
test = df_all[df_all.Date >= "2023-01-01"]

print(train.Date.min(), train.Date.max())
print(valid.Date.min(), valid.Date.max())
print(test.Date.min(), test.Date.max())

2016-06-08 00:00:00-04:00 2021-12-31 00:00:00-05:00
2022-01-03 00:00:00-05:00 2022-12-30 00:00:00-05:00
2023-01-03 00:00:00-05:00 2024-11-15 00:00:00-05:00


In [77]:
X_train = train[features]
Y_train = train["target"]

X_valid = valid[features]
Y_valid = valid["target"]

X_test = test[features]
Y_test = test["target"]

# use scalar on X
from sklearn.preprocessing import StandardScaler
scalar = StandardScaler()
X_train_scaled = scalar.fit_transform(X_train)
X_valid_scaled = scalar.fit_transform(X_valid)
X_test_scaled = scalar.fit_transform(X_test)

# now fit model to training set
from sklearn.linear_model import Ridge
ridge = Ridge(alpha=0.4)
ridge.fit(X_train_scaled, Y_train)
print("Model trained successfully, predicting from validation set...")
y_pred_ridge = ridge.predict(X_valid_scaled)

from sklearn.metrics import mean_squared_error
rmse_ridge = mean_squared_error(Y_valid, y_pred_ridge) ** 0.5
print(rmse_ridge)


Model trained successfully, predicting from validation set...
0.15554987369915016


In [76]:
baseline_pred = Y_train.mean()
rmse_baseline = mean_squared_error(Y_valid, [baseline_pred] * len(Y_valid)) ** 0.5
print(rmse_baseline)

0.15127820925011765


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbreg = GradientBoostingRegressor()
gbreg.fit(X_train, Y_train)
y_pred_gb = gbreg.predict(X_valid)
rmse_gb = mean_squared_error(Y_valid, y_pred_gb) ** 0.5
print(rmse_gb)

/Users/willgladstone_/Documents/Learning ML for Quant/quant_ML/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(


ValueError: Expected a 2-dimensional container but got <class 'pandas.Series'> instead. Pass a DataFrame containing a single row (i.e. single sample) or a single column (i.e. single feature) instead.